### Подготовим данные для дообучения.

https://github.com/meta-llama/synthetic-data-kit

In [1]:
# @title
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm synthetic-data-kit==0.0.3
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade         unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
    !uv pip install synthetic-data-kit==0.0.3
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [ ]:
from unsloth.dataprep import SyntheticDataKit

generator = SyntheticDataKit.from_pretrained(
    # Choose any model from https://huggingface.co/unsloth
    model_name = "unsloth/Qwen2-0.5B-Instruct",
    max_seq_length = 2048, # Longer sequence lengths will be slower!
)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# @title
!synthetic-data-kit system-check

vLLM STDOUT: INFO:     127.0.0.1:52834 - "GET /v1/models HTTP/1.1" 200 OK
 VLLM server is running at http://localhost:8000/v1
Available models: {'object': 'list', 'data': [{'id': 
'unsloth/Qwen2-0.5B-Instruct', 'object': 'model', 'created': 1758341554, 
'owned_by': 'vllm', 'root': 'unsloth/Qwen2-0.5B-Instruct', 'parent': None, 
'max_model_len': 2048, 'permission': [{'id': 
'modelperm-0d9204f4f8094e939c2b12c7b57c5636', 'object': 'model_permission', 
'created': 1758341554, 'allow_create_engine': False, 'allow_sampling': True, 
'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 
'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': 
False}]}]}
⠋ Checking VLLM server at http://localhost:8000/v1...


In [4]:
import json

input_file = "/content/drive/MyDrive/qwen_funetune/json/parsed_data.json"
output_file = "/content/drive/MyDrive/qwen_funetune/json/parsed_data.txt"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

data = data[:5]

with open(output_file, "w", encoding="utf-8") as f:
    for item in data:
        lines = []
        for key, value in item.items():
            lines.append(f"{key}: {value}")
        f.write("\n".join(lines))
        f.write("\n" + "="*40 + "\n\n")


In [6]:
generator.prepare_qa_generation(
    output_folder = 'data', # Output location of synthetic data
    temperature = 0.7, # Higher temp makes more diverse datases
    top_p = 0.95,
    overlap = 64, # Overlap portion during chunking
    max_generation_tokens = 512, # Can increase for longer QA pairs
)

## Document Parsing (txt)

In [8]:
# 1️⃣ Ingest
!synthetic-data-kit ingest /content/drive/MyDrive/qwen_funetune/parsed/parsed_data.txt

⠋ Processing /content/drive/MyDrive/qwen_funetune/parsed/parsed_data.txt...
 Text successfully extracted to data/output/parsed_data.txt


In [15]:
!synthetic-data-kit \
    -c synthetic_data_kit_config.yaml \
    ingest /content/drive/MyDrive/qwen_funetune/parsed/parsed_data.txt

# Truncate document
filenames = generator.chunk_data("data/output/parsed_data.txt")
print(len(filenames), filenames[:3])

⠋ Processing /content/drive/MyDrive/qwen_funetune/parsed/parsed_data.txt...
 Text successfully extracted to data/output/parsed_data.txt
42 ['data/output/parsed_data_0.txt', 'data/output/parsed_data_1.txt', 'data/output/parsed_data_2.txt']


In [ ]:
import time
# Process 3 chunks for now -> can increase but slower!
for filename in filenames:
    !synthetic-data-kit \
        -c synthetic_data_kit_config.yaml \
        create {filename} \
        --num-pairs 25 \
        --type "qa"
    time.sleep(2) # Sleep some time to leave some room for processing

In [22]:
qa_pairs_filenames = [
    f"data/generated/parsed_data_{i}_qa_pairs.json"
    for i in range(len(filenames))
]
for filename in qa_pairs_filenames:
    !synthetic-data-kit \
        -c synthetic_data_kit_config.yaml \
        save-as {filename} -f ft

⠋ Converting data/generated/parsed_data_0_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data_0_qa_pairs_ft.json
⠋ Converting data/generated/parsed_data_1_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data_1_qa_pairs_ft.json
⠋ Converting data/generated/parsed_data_2_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data_2_qa_pairs_ft.json
⠋ Converting data/generated/parsed_data_3_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data_3_qa_pairs_ft.json
⠋ Converting data/generated/parsed_data_4_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data_4_qa_pairs_ft.json
⠋ Converting data/generated/parsed_data_5_qa_pairs.json to ft format with json 
storage...
 Converted to ft format and saved to data/final/parsed_data

In [23]:
from datasets import Dataset
import pandas as pd

final_filenames = [
    f"data/final/parsed_data_{i}_qa_pairs_ft.json"
    for i in range(len(filenames[:3]))
]
conversations = pd.concat([
    pd.read_json(name) for name in final_filenames
]).reset_index(drop = True)

dataset = Dataset.from_pandas(conversations)

In [24]:
dataset[0]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': "What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?",
   'role': 'user'},
  {'content': '$599', 'role': 'assistant'}]}

In [34]:
# как Dataset (для восстановления через load_from_disk)
dataset.save_to_disk("/content/drive/MyDrive/qwen_funetune/final/dataset_items")

# как JSON (для простого доступа к данным)
dataset.to_json("/content/drive/MyDrive/qwen_funetune/final/dataset_items.json")

Saving the dataset (0/1 shards):   0%|          | 0/35 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

22989

In [26]:
generator.cleanup()

Attempting to terminate the VLLM server gracefully...
Server terminated gracefully.
